In [ ]:
import time
from pathlib import Path
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.b1500 import B1500, B1500Error


cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

# 第一组候选映射：先按这个试
CH_G = 4
CH_D = 5
CH_S = 6

# 小信号 sanity 用
I_COMP = 1e-3
VRANGE_G = 0
VRANGE_D = 0
VRANGE_S = 0

print("候选映射：")
print(f"G = CH{CH_G}, D = CH{CH_D}, S = CH{CH_S}")


session = VisaSession(cfg)
session.open()
b = B1500(session)

print("IDN:", session.query("*IDN?"))
print("ERRX:", b.errx())

def safe_zero_and_cl(channels):
    try:
        b.dz(channels)
    except Exception as e:
        print("DZ warning:", e)

    try:
        b.cl(channels)
    except Exception as e:
        print("CL warning:", e)

In [ ]:
static_sets = [
    {"name": "set1", "vg": 0.0,  "vd": 0.0,  "vs": 0.0},
    {"name": "set2", "vg": -0.1, "vd": 0.05, "vs": 0.0},
    {"name": "set3", "vg": -0.2, "vd": 0.10, "vs": 0.0},
    {"name": "set4", "vg": 0.0,  "vd": 0.10, "vs": 0.0},
]
pd.DataFrame(static_sets)

for s in static_sets:
    print(f"\n===== {s['name']} =====")
    print(f"设定: Vg={s['vg']} V, Vd={s['vd']} V, Vs={s['vs']} V")

    b.cn([CH_G, CH_D, CH_S])

    b.dv(CH_G, VRANGE_G, s["vg"], I_COMP)
    b.dv(CH_D, VRANGE_D, s["vd"], I_COMP)
    b.dv(CH_S, VRANGE_S, s["vs"], I_COMP)

    time.sleep(0.5)

    print("请现在用示波器/万用表核对三端实际电压。")
    print("确认后，再手动运行下一组，或最后运行 Cell 7 归零。")

    safe_zero_and_cl([CH_G, CH_D, CH_S])
print("已归零并关闭三端。")